In [1]:
import pandas as pd

DATA_PATH = "../data/ai4i2020.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Baseline model: Majority Class

Since most machine do not fail, a majority class model will always predict no failure.

This baseline is useful because any real model should beat it on recall and F1 for sure.

In [2]:
target_col = "Machine failure"

feature_col = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

X = df[feature_col]
y = df[target_col]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [4]:
X = df[feature_col]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

Used a stratified train/test split so the failure rate stays similar in both sets. Matters because the +ve class is rare.

In [28]:
dummy_model = DummyClassifier(strategy = "most_frequent")
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

In [29]:
def evaluate_classification_model(y_true, y_pred):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    matrix = confusion_matrix(y_true, y_pred)

    return metrics, matrix

In [33]:
dummy_metrics, dummy_matrix = evaluate_classification_model(y_test, dummy_predictions)
print(dummy_metrics,"\n")
print(dummy_matrix)

{'accuracy': 0.966, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0} 

[[1932    0]
 [  68    0]]


The dummy model has high accuracy because almost all machine do not fail. However, the recall is 0, this means it did not catch any actual failures.

## Logistic regression

The dummy model did not learn from the sensor features. Now lets train logistic regression model.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

The dataset has one categorical feature i.e. Type.
For logistic regression:
- one-hot encode "Type"
- Scale numeric features
- use class_weight="balanced" because failures are rare

In [8]:
categorical_features = ["Type"]
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

In [9]:
pre_processing = ColumnTransformer(
    transformers=[
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ]
)
pre_processing

,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [10]:
logistic_model = Pipeline(
    steps=[
        ("pre_processing", pre_processing),
        ("classifier", LogisticRegression(class_weight="balanced",max_iter=1000))
    ]
)
logistic_model

,steps,"[('pre_processing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
logistic_model.fit(X_train,y_train)

,steps,"[('pre_processing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
logistic_predictions = logistic_model.predict(X_test)

In [35]:
logistic_metrics, logistic_matrix = evaluate_classification_model(y_test, logistic_predictions)
print(logistic_metrics,'\n')
print(logistic_matrix)

{'accuracy': 0.8245, 'precision': 0.14177215189873418, 'recall': 0.8235294117647058, 'f1': 0.24190064794816415} 

[[1593  339]
 [  12   56]]


Logistic regression model catches many more failures than the dummy model.

From the confusion matrix:
- It catches 56 actual failures.
- It misses 12 failures.
- It creates 339 false alarms.

This is a typical imbalance classification tradeoff. The model has much better recall than the dummy model, but precision is low because it predicts many failures that do not actually happen.

## Random Forest baseline

Logistic regression is a linear model. A random forest is a tree-based model that can capture nonlinear patterns.

In [15]:
from sklearn.ensemble import RandomForestClassifier

In [17]:
rf_preprocessing = ColumnTransformer(
    transformers=[
        ("category",OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough",numeric_features)
    ]
)
rf_preprocessing

,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [19]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessing", rf_preprocessing),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42)
        )
    ]
)
random_forest_model

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
random_forest_model.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('category', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
rf_predictions = random_forest_model.predict(X_test)

In [36]:
rf_metrics, rf_matrix = evaluate_classification_model(y_test, rf_predictions)
print(rf_metrics,"\n")
print(rf_matrix)

{'accuracy': 0.98, 'precision': 0.9375, 'recall': 0.4411764705882353, 'f1': 0.6} 

[[1930    2]
 [  38   30]]


Random forest has much higher precision than logistic regression, meaning fewer false alarms.
However, its recall is lower, so it misses more actual failures.
This shoes that model choice changes the maintenance tradeoff: one model may be better when false alarms are expensive, while another may be better when missed failures are more expensive.

In [37]:
pd.DataFrame(
    [dummy_metrics, logistic_metrics, rf_metrics],
    index=["Dummy", "Logistic Regression", "Random Forest"]
)

,accuracy,precision,recall,f1
Dummy,0.9660,0.000000,0.000000,0.000000
Logistic Regression,0.8245,0.141772,0.823529,0.241901
Random Forest,0.9800,0.937500,0.441176,0.600000
